# DSA 8301 — Statistical Inference for Big Data
## Kenya Housing Survey 2023/24 — Data Cleaning & Preparation (v2)

**Student:** Valerie Jerono | **Reg No:** 222331  
**Course:** DSA 8301 — Statistical Inference for Big Data  
**Lecturer:** Prof. Jacob Ong'ala  
**Institution:** Strathmore University | iLabAfrica Research Centre  
**Date:** June 2026  

---

### Purpose & Design Philosophy of this Notebook

This notebook is a full redesign of Phase 2 (Preprocessing + Feature Engineering). The previous version dropped too many variables en route to the working subset, leaving the final model's pillar construction under-resourced. This version takes the opposite approach:

> **Principle:** Clean *everything that the model needs*, engineer *every pillar ingredient and proxy feature*, and only then reduce to the target 25–35 variable analytical frame — with every exclusion explicitly justified.

**Five pillars of the Housing Financial Vulnerability Score (HFVS):**
| Pillar | Weight | Core variables |
|---|---|---|
| P1 Financial Stress | 0.30 | `k05`, `g01a`–`g01k`, `j09`, `j10` |
| P2 Physical Quality | 0.20 | `d08`/`dw_wall_mat`, `d09`/`dw_roof_mat`, `d10`/`dw_floor_mat`, `c04`, `c01_1` |
| P3 Tenure Security | 0.20 | `j05`, `j11`, `i00`, `j04_1` |
| P4 Hazard Exposure | 0.15 | `e06`, `e07`, `d07` |
| P5 Utility Deprivation | 0.15 | `c01_1`, `c04`, `c11`, `c10`, `c05` |

**Proxy feature matrix (24 variables, no leakage):** demographic + geographic + asset + stated-problem + aspiration variables that the model uses to *predict* HFVS class without using any formula ingredient.

**Final analytical frame target:** 25–35 columns covering all 5 pillars + all proxy features + essential demographics + geographic identifiers.

---
### Workflow
1. **Setup & Load** — re-establish environment, load `master_frame.parquet`
2. **Initial Audit** — shape, types, full missingness scan (BEFORE picture)
3. **Sentinel & Plausibility Scan** — replace invalid codes with NaN *before* tiering
4. **Missingness Tiering** — classify every column into Complete / Low / Moderate / High
5. **Structural Missingness** — encode meaningful NaN (land parcels, renters, county-finance)
6. **Provenance Verification** — check `prop_util`, `med_brms`, `min_rent`, `med_prop`
7. **General Imputation** — stratified group-median for Low/Moderate tiers
8. **Outlier Treatment** — winsorise monetary variables at [1st, 99th pct] by stratum
9. **Consistency Checks** — cross-variable logic audits
10. **Feature Engineering** — all 5 pillar components + all 24 proxy features + interaction terms
11. **Column Inventory & Selection** — build the final 25–35 variable frame with justification table
12. **Post-Cleaning EDA** — distributions, urban/rural splits, correlations
13. **Save** — `master_frame_clean.parquet` + `dsa8301_working_subset.parquet`
14. **Cleaning Summary Table** — preprocessing log for the report


---
## 0. Setup & Load

In [ ]:
# ── 0.1  Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 0.2  Core imports ─────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.stats import skew

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 120)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
})
sns.set_style('whitegrid')

TEAL   = '#00695C'; RED  = '#B71C1C'; AMBER = '#E65100'
BLUE   = '#1565C0'; GRAY = '#546E7A'; GREEN = '#2E7D32'

print('All imports loaded.')

In [ ]:
# ── 0.3  Paths ────────────────────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]: p.mkdir(parents=True, exist_ok=True)

MASTER_PATH = PQ / 'master_frame.parquet'
print(f'Paths configured. master_frame.parquet exists: {MASTER_PATH.exists()}')

In [ ]:
# ── 0.4  Load the master frame (produced by exploration notebook) ─────────
if not MASTER_PATH.exists():
    # Fallback: try clean version if it exists
    alt = PQ / 'master_frame_clean.parquet'
    if alt.exists():
        MASTER_PATH = alt
    else:
        raise FileNotFoundError(
            'master_frame.parquet not found. Re-run the exploration notebook '
            'and add master_clean.to_parquet(PQ / "master_frame.parquet") '
            'at the end of Section 1.10.'
        )

master = pd.read_parquet(MASTER_PATH)
print(f'Loaded master frame: {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'\nColumn groups present (prefix scan):')
for prefix in ['hh_', 'county_', 'a0', 'b0', 'c0', 'c1', 'd0', 'd1', 'e0',
               'g0', 'h0', 'i0', 'j0', 'k0', 'l0', 'lp_', 'dw_', 'mort_',
               'fin_', 'loan_', 'wsvc_', 'nema_']:
    cols = [c for c in master.columns if c.startswith(prefix)]
    if cols:
        print(f'  {prefix:<10} {len(cols):>3} columns')

In [ ]:
# ── 0.5  Column name normalisation ────────────────────────────────────────
# The exploration notebook applied RENAME_MAP. Detect which naming convention
# is present and create a unified accessor dictionary so all downstream cells
# work regardless of whether raw (a01) or clean (county_code) names are used.

# Key column aliases: raw_name -> clean_name (from exploration RENAME_MAP)
ALIAS = {
    'a01'       : 'county_code',
    'a07_1'     : 'urban_rural',
    'hhweight'  : 'hh_weight',
    'hhsize'    : 'hh_size',
    'hh_size'   : 'hh_size',      # already clean
    'c01_1'     : 'water_src_main',
    'c04'       : 'toilet_type',
    'c05'       : 'has_handwash_facility',
    'c10'       : 'lighting_src',
    'c11'       : 'cooking_fuel',
    'c13__2'    : 'owns_mobile',
    'c13__3'    : 'owns_tv',
    'internet'  : 'has_internet',
    'g02'       : 'pays_rent',
    'g03'       : 'tenure_type',
    'g04'       : 'owns_other_property',
    'g05__1'    : 'prob_overcrowding',
    'g05__2'    : 'prob_poor_water',
    'g05__7'    : 'prob_high_cost',
    'h11'       : 'perc_overall',
    'i00'       : 'owns_land',
    'j04_1'     : 'is_owner_occupier',
    'j05'       : 'has_title_doc',
    'j09'       : 'housing_cost_burden',
    'j10'       : 'missed_payment',
    'j11'       : 'eviction_risk',
    'j12_1'     : 'yrs_in_dwelling',
    'j13'       : 'satisfied_tenure',
    'k05'       : 'rent_actual_kes',
    'e06'       : 'flood_exposure',
    'e07'       : 'erosion_exposure',
    'd07'       : 'dw_in_hazard_zone',
    'sf'        : 'is_slum',
}

def col(name):
    """Return the actual column name in master, checking alias both ways."""
    if name in master.columns:
        return name
    for raw, clean in ALIAS.items():
        if name == clean and raw in master.columns:
            return raw
        if name == raw and clean in master.columns:
            return clean
    return name  # will raise KeyError downstream if truly missing

# Normalise a handful of critical column names for downstream cells
RENAMES = {}
for raw, clean in ALIAS.items():
    if raw in master.columns and clean not in master.columns:
        RENAMES[raw] = clean

if RENAMES:
    master = master.rename(columns=RENAMES)
    print(f'Renamed {len(RENAMES)} columns to clean names: {list(RENAMES.values())[:8]}...')
else:
    print('Clean column names already present — no renaming needed.')

print(f'\nFinal shape after normalisation: {master.shape}')

---
## 1. Initial Audit — BEFORE Picture

In [ ]:
# ── 1.1  Shape, types, duplicates ─────────────────────────────────────────
print(f'Shape          : {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'Duplicate rows : {master.duplicated().sum()}')
id_col = col('hh_id') if 'hh_id' in master.columns else 'interview__key'
if id_col in master.columns:
    print(f'Duplicate IDs  : {master[id_col].duplicated().sum()}')

print(f'\nDtype breakdown:')
print(master.dtypes.value_counts())

In [ ]:
# ── 1.2  Full missingness scan — BEFORE cleaning ──────────────────────────
miss_before = pd.DataFrame({
    'n_missing'  : master.isnull().sum(),
    'pct_missing': (master.isnull().mean() * 100).round(2)
}).sort_values('pct_missing', ascending=False)

print(f'Columns with   0% missing : {(miss_before.pct_missing == 0).sum()}')
print(f'Columns with  >0% missing : {(miss_before.pct_missing > 0).sum()}')
print(f'Columns with >20% missing : {(miss_before.pct_missing > 20).sum()}')
print(f'Columns with >60% missing : {(miss_before.pct_missing > 60).sum()}')
print()
print('Top 25 most-missing columns:')
print(miss_before.head(25))

In [ ]:
# ── 1.3  Confirm pillar columns are present ───────────────────────────────
# These are the non-negotiable columns for HFVS construction.
# If any are missing, flag it now before spending time on cleaning.

PILLAR_REQUIRED = {
    'P1_Financial'  : ['k05', 'rent_actual_kes', 'g01a', 'g01b', 'g01c', 'g01d',
                        'g01e', 'g01f', 'g01g', 'g01h', 'g01i', 'g01j', 'g01k',
                        'j09', 'housing_cost_burden', 'j10', 'missed_payment'],
    'P2_Physical'   : ['dw_wall_mat', 'd08', 'dw_roof_mat', 'd09',
                        'dw_floor_mat', 'd10', 'c04', 'toilet_type',
                        'c01_1', 'water_src_main'],
    'P3_Tenure'     : ['j05', 'has_title_doc', 'j11', 'eviction_risk',
                        'i00', 'owns_land', 'j04_1', 'is_owner_occupier'],
    'P4_Hazard'     : ['e06', 'flood_exposure', 'e07', 'erosion_exposure',
                        'd07', 'dw_in_hazard_zone'],
    'P5_Utility'    : ['c01_1', 'water_src_main', 'c04', 'toilet_type',
                        'c11', 'cooking_fuel', 'c10', 'lighting_src',
                        'c05', 'has_handwash_facility'],
}

print('PILLAR COLUMN AVAILABILITY:')
print('=' * 60)
for pillar, cols in PILLAR_REQUIRED.items():
    present = [c for c in cols if c in master.columns]
    missing = [c for c in cols if c not in master.columns]
    # At least one alias must resolve per concept
    print(f'  {pillar:<18} {len(present):>2}/{len(cols):>2} found', end='')
    if missing:
        print(f'  ⚠ Missing: {missing[:4]}')
    else:
        print(' ✓')

# Resolve which exact names to use for each pillar concept
# (prefer clean name, fall back to raw)
WALL_COL   = 'dw_wall_mat'  if 'dw_wall_mat'  in master.columns else 'd08'
ROOF_COL   = 'dw_roof_mat'  if 'dw_roof_mat'  in master.columns else 'd09'
FLOOR_COL  = 'dw_floor_mat' if 'dw_floor_mat' in master.columns else 'd10'
TOILET_COL = 'toilet_type'  if 'toilet_type'  in master.columns else 'c04'
WATER_COL  = 'water_src_main' if 'water_src_main' in master.columns else 'c01_1'
COOK_COL   = 'cooking_fuel' if 'cooking_fuel'  in master.columns else 'c11'
LIGHT_COL  = 'lighting_src' if 'lighting_src'  in master.columns else 'c10'
WASH_COL   = 'has_handwash_facility' if 'has_handwash_facility' in master.columns else 'c05'
FLOOD_COL  = 'flood_exposure' if 'flood_exposure' in master.columns else 'e06'
ERODE_COL  = 'erosion_exposure' if 'erosion_exposure' in master.columns else 'e07'
HAZ_COL    = 'dw_in_hazard_zone' if 'dw_in_hazard_zone' in master.columns else 'd07'
TITLE_COL  = 'has_title_doc' if 'has_title_doc' in master.columns else 'j05'
EVICT_COL  = 'eviction_risk' if 'eviction_risk' in master.columns else 'j11'
LAND_COL   = 'owns_land'     if 'owns_land'     in master.columns else 'i00'
OWNER_COL  = 'is_owner_occupier' if 'is_owner_occupier' in master.columns else 'j04_1'
RENT_COL   = 'rent_actual_kes'   if 'rent_actual_kes'   in master.columns else 'k05'
BURDEN_COL = 'housing_cost_burden' if 'housing_cost_burden' in master.columns else 'j09'
MISSED_COL = 'missed_payment'   if 'missed_payment'   in master.columns else 'j10'
URBAN_COL  = 'urban_rural'      if 'urban_rural'       in master.columns else 'a07_1'
COUNTY_COL = 'county_code'      if 'county_code'       in master.columns else 'a01'
WEIGHT_COL = 'hh_weight'        if 'hh_weight'         in master.columns else 'hhweight'
SLUM_COL   = 'is_slum'          if 'is_slum'           in master.columns else 'sf'

EXP_COLS = [c for c in ['g01a','g01b','g01c','g01d','g01e',
                          'g01f','g01g','g01h','g01i','g01j','g01k']
            if c in master.columns]
print(f'\nExpenditure columns present: {len(EXP_COLS)}/11 → {EXP_COLS}')

---
## 2. Sentinel & Plausibility Scan

Survey instruments encode "Don't know", "Refused", and "Not applicable" as numeric sentinels (8, 9, 98, 99, 998, 999, -99). These **must be replaced with NaN before any missingness tiering**, otherwise the tier classification itself is wrong.

In [ ]:
# ── 2.1  Plausibility scan — continuous variables ─────────────────────────
PLAUS_CHECKS = {
    'b05_years'  : (0, 110),    # age in years
    'mean_age'   : (0, 110),
    'l07'        : (1900, 2026), # year built
    'dwelling_yr_built': (1900, 2026),
    'c01_4'      : (0, 600),    # minutes to water source
    'water_dist_mins': (0, 600),
    'c10_2'      : (0, 24),     # hours of electricity/day
    'electricity_hrs_day': (0, 24),
}

# Expenditure columns — must be >= 0
for c in EXP_COLS + [RENT_COL, 'l13', 'mortgage_repayment_kes', 'l14',
                      'dwelling_value_kes', 'c14_1', 'spend_water_kes',
                      'c14_2', 'spend_electricity_kes', 'c14_3']:
    if c in master.columns:
        PLAUS_CHECKS[c] = (0, None)

plaus_report = []
for c, (lo, hi) in PLAUS_CHECKS.items():
    if c not in master.columns:
        continue
    s = pd.to_numeric(master[c], errors='coerce')
    n_lo = (s < lo).sum()  if lo is not None else 0
    n_hi = (s > hi).sum()  if hi is not None else 0
    if n_lo or n_hi:
        plaus_report.append({'variable': c, 'rule': f'[{lo},{hi}]',
                             'n_below': n_lo, 'n_above': n_hi,
                             'min': s.min(), 'max': s.max()})

plaus_df = pd.DataFrame(plaus_report)
print(f'Plausibility violations found: {len(plaus_df)}')
print(plaus_df)

In [ ]:
# ── 2.2  Sentinel-code scan — ordinal & categorical variables ─────────────
SENTINEL_CODES = {8, 9, 98, 99, 998, 999, -99, -98}

# Variables with small expected value sets — scan for outlier codes
SMALL_VALUE_COLS = [
    TOILET_COL, WATER_COL, COOK_COL, LIGHT_COL, WASH_COL,
    FLOOD_COL, ERODE_COL, 'e08', 'terrain_type',
    'd03', 'dw_type', 'd01', 'hh_tenure_type',
    URBAN_COL, OWNER_COL, TITLE_COL, EVICT_COL, BURDEN_COL, MISSED_COL,
    'h01','h02','h03','h04','h05','h06','h07','h08','h09','h10','h11',
    'perc_overall', 'perc_structure', 'perc_roof', 'perc_walls',
    'g03', 'tenure_type',
]

sentinel_report = []
for c in SMALL_VALUE_COLS:
    if c not in master.columns:
        continue
    vc = master[c].value_counts(dropna=True)
    found = SENTINEL_CODES.intersection(set(vc.index))
    for code in found:
        sentinel_report.append({'variable': c, 'sentinel': code,
                                 'n_rows': vc[code],
                                 'pct': round(vc[code]/vc.sum()*100, 2)})

sentinel_df = (pd.DataFrame(sentinel_report).sort_values('n_rows', ascending=False)
               if sentinel_report else
               pd.DataFrame(columns=['variable','sentinel','n_rows','pct']))
print(f'Sentinel codes found in {sentinel_df.variable.nunique()} variables:')
print(sentinel_df)

In [ ]:
# ── 2.3  Apply corrections ────────────────────────────────────────────────
CORRECTIONS_LOG = []  # (variable, action, n_affected)

# --- 2.3a  Plausibility: cap implausible continuous values ----------------
# Year built: anything outside [1900, 2026] is a likely entry error
for c in ['l07', 'dwelling_yr_built']:
    if c in master.columns:
        n = ((master[c] < 1900) | (master[c] > 2026)).sum()
        if n:
            master.loc[(master[c] < 1900) | (master[c] > 2026), c] = np.nan
            CORRECTIONS_LOG.append((c, 'year outside [1900,2026] → NaN', n))

# Hours of electricity: > 24 is impossible
for c in ['c10_2', 'electricity_hrs_day']:
    if c in master.columns:
        n = (master[c] > 24).sum()
        if n:
            master.loc[master[c] > 24, c] = np.nan
            CORRECTIONS_LOG.append((c, 'hours > 24 → NaN', n))

# Negative expenditure: floor at 0
for c in EXP_COLS:
    if c in master.columns:
        n = (master[c] < 0).sum()
        if n:
            master.loc[master[c] < 0, c] = 0
            CORRECTIONS_LOG.append((c, 'negative spend → 0', n))

# --- 2.3b  Sentinel codes → NaN ------------------------------------------
for _, row in sentinel_df.iterrows():
    c = row['variable']
    code = row['sentinel']
    if c in master.columns:
        n = (master[c] == code).sum()
        master.loc[master[c] == code, c] = np.nan
        CORRECTIONS_LOG.append((c, f'sentinel {code} → NaN', n))

print(f'Corrections applied: {len(CORRECTIONS_LOG)}')
corr_df = pd.DataFrame(CORRECTIONS_LOG, columns=['variable', 'action', 'n_affected'])
print(corr_df)

---
## 3. Missingness Tiering (Post-Sentinel)

| Tier | Missing % | Default treatment |
|---|---|---|
| Complete | 0% | Use as-is |
| Low | 1–20% | Stratified group-median/mode imputation |
| Moderate | 21–60% | Same, plus `_was_imputed` flag |
| High | >60% | Do NOT impute generically — encode structurally where meaningful, exclude otherwise |

In [ ]:
# ── 3.1  Re-scan missingness post-sentinel-correction & assign tiers ──────
miss_post = (master.isnull().mean() * 100).round(2)

def tier(pct):
    if pct == 0:          return 'Complete'
    elif pct <= 20:       return 'Low'
    elif pct <= 60:       return 'Moderate'
    else:                 return 'High'

tier_df = pd.DataFrame({'pct_missing': miss_post})
tier_df['tier'] = tier_df['pct_missing'].apply(tier)

print('Tier counts (post sentinel-correction):')
print(tier_df['tier'].value_counts())
print()
print('HIGH-tier columns (>60% missing) — names only:')
high_cols = tier_df[tier_df['tier'] == 'High'].index.tolist()
for c in high_cols:
    print(f'  {c:<45} {tier_df.loc[c, "pct_missing"]:>6.1f}%')

In [ ]:
# ── 3.2  Identify pillar columns that are in the High tier ────────────────
# These must NOT be dropped — structural encoding takes priority over tier.
ALL_PILLAR_COLS = [
    RENT_COL, BURDEN_COL, MISSED_COL,           # P1
    WALL_COL, ROOF_COL, FLOOR_COL,              # P2 materials
    TOILET_COL, WATER_COL,                       # P2 utilities
    TITLE_COL, EVICT_COL, LAND_COL, OWNER_COL,  # P3
    FLOOD_COL, ERODE_COL, HAZ_COL,              # P4
    COOK_COL, LIGHT_COL, WASH_COL,              # P5
] + EXP_COLS
ALL_PILLAR_COLS = list(set(c for c in ALL_PILLAR_COLS if c in master.columns))

pillar_high = [c for c in ALL_PILLAR_COLS if tier_df.get('pct_missing', {}).get(c, 0) > 60]
print(f'Pillar columns in High tier: {pillar_high}')
print('These will be structurally encoded in Section 4, not dropped.')

---
## 4. Structural Missingness — Encode, Do Not Impute

Three families of columns have missingness that is **meaningful, not random**. Imputing them with group median would be substantively wrong.

In [ ]:
# ── 4.1  Verify: lp_* missingness ↔ i00/owns_land == 0 ──────────────────
lp_cols = [c for c in master.columns if c.startswith('lp_')]
print(f'Land-parcel columns (lp_*): {lp_cols}')

if lp_cols and LAND_COL in master.columns:
    miss_mask = master[lp_cols[0]].isnull()
    ct = pd.crosstab(master[LAND_COL], miss_mask,
                     rownames=[LAND_COL], colnames=[f'{lp_cols[0]} is null'])
    print(ct)
    agree = (master[LAND_COL].eq(0) == miss_mask).mean()
    print(f'Agreement (land==0 ↔ lp_null): {agree:.2%}')
print()

In [ ]:
# ── 4.2  Verify: rent_actual_kes null ↔ non-renter ────────────────────────
pays_rent_col = 'pays_rent' if 'pays_rent' in master.columns else 'g02'
if RENT_COL in master.columns and pays_rent_col in master.columns:
    rent_null = master[RENT_COL].isnull()
    not_renter = (master[pays_rent_col] != 1)
    print(f'Rent null     : {rent_null.sum():,} ({rent_null.mean():.1%})')
    print(f'Not renter    : {not_renter.sum():,} ({not_renter.mean():.1%})')
    overlap = (rent_null & not_renter).sum()
    print(f'Both (expected structural): {overlap:,}')
    spurious = (master[RENT_COL].notna() & not_renter).sum()
    print(f'Non-renters with rent > 0 (consistency issue): {spurious}')

In [ ]:
# ── 4.3  Encode all structural missingness ────────────────────────────────

# (a) Land-parcel columns — null means 'no land'
for c in lp_cols:
    if c not in master.columns:
        continue
    if c == 'lp_n_parcels':
        master[c] = pd.to_numeric(master[c].replace('No land', 0), errors='coerce').fillna(0)
    elif master[c].dtype in ['float64', 'int64'] or str(master[c].dtype).startswith('Int'):
        # Binary/numeric: 0 = no land
        master[c] = master[c].fillna(0)
    else:
        master[c] = master[c].fillna('No land').astype(str)

print(f'lp_* encoded: {len(lp_cols)} columns, null → 0 / "No land"')

# (b) is_renter flag — binary flag so downstream cells can filter
if pays_rent_col in master.columns:
    master['is_renter'] = (master[pays_rent_col] == 1).astype('Int64')
else:
    # Infer from rent amount
    master['is_renter'] = (master[RENT_COL].notna() & (master[RENT_COL] > 0)).astype('Int64') if RENT_COL in master.columns else 0
print(f'is_renter: {master["is_renter"].sum():,} renters ({master["is_renter"].mean():.1%})')

# k05/rent_actual_kes: leave as NaN for non-renters (structural, correct)
# Correct spurious non-renter rent values
if RENT_COL in master.columns and pays_rent_col in master.columns:
    spurious_mask = (master[pays_rent_col] != 1) & (master[RENT_COL].notna())
    n_spur = spurious_mask.sum()
    if n_spur:
        master.loc[spurious_mask, RENT_COL] = np.nan
        CORRECTIONS_LOG.append((RENT_COL, f'non-renter with rent → NaN', n_spur))

# (c) County-finance columns (mort_*, fin_*, loan_*) — null = no formal market
mort_fin_cols = [c for c in master.columns if c.startswith(('mort_', 'fin_', 'loan_'))]
for c in mort_fin_cols:
    flag = f'{c}_market_absent'
    master[flag] = master[c].isnull().astype(int)
    master[c]   = master[c].fillna(0)

print(f'mort_/fin_/loan_ encoded: {len(mort_fin_cols)} columns, _market_absent flags created')

---
## 5. Provenance Verification — Pre-Computed Columns

Four columns flagged as "provenance unverified" in `Data_Understanding.md`: `prop_util`, `med_prop`, `med_brms`, `min_rent`. Each hypothesis is tested here.

In [ ]:
# ── 5.1  Verify prop_util: utilities / total_exp ──────────────────────────
util_cols = [c for c in ['c14_1','c14_2','c14_3','spend_water_kes',
                           'spend_electricity_kes','spend_energy_other_kes']
             if c in master.columns]

utility_exp = master[util_cols].sum(axis=1, skipna=True)
total_exp_cand = master[EXP_COLS].sum(axis=1, skipna=True)
cand_a = utility_exp / total_exp_cand.replace(0, np.nan)

if 'prop_util' in master.columns:
    valid = master['prop_util'].notna()
    corr = master.loc[valid, 'prop_util'].corr(cand_a[valid])
    print(f'prop_util: n={valid.sum():,}, corr vs utilities/total_exp = {corr:.3f}')
    PROP_UTIL_VERIFIED = corr > 0.7
    print(f'  → Verdict: {"VERIFIED" if PROP_UTIL_VERIFIED else "UNVERIFIED"} (threshold: r > 0.7)')
elif 'util_income_ratio' in master.columns:
    valid = master['util_income_ratio'].notna()
    corr = master.loc[valid, 'util_income_ratio'].corr(cand_a[valid])
    print(f'util_income_ratio: n={valid.sum():,}, corr vs utilities/total_exp = {corr:.3f}')
    PROP_UTIL_VERIFIED = corr > 0.7
else:
    # Construct it ourselves
    master['prop_util'] = cand_a
    PROP_UTIL_VERIFIED = True
    print('prop_util not found — constructed from utility_exp / total_exp_cand')

In [ ]:
# ── 5.2  Verify med_brms: county median of bedrooms ───────────────────────
bedroom_col = 'dw_bedrooms' if 'dw_bedrooms' in master.columns else 'd11_2'
med_brms_col = 'cty_med_bedrooms' if 'cty_med_bedrooms' in master.columns else 'med_brms'

if med_brms_col in master.columns and bedroom_col in master.columns:
    county_med = master.groupby(COUNTY_COL)[bedroom_col].median()
    mapped = master[COUNTY_COL].map(county_med)
    valid = master[med_brms_col].notna()
    match = (master.loc[valid, med_brms_col] == mapped[valid]).mean()
    print(f'{med_brms_col}: n={valid.sum():,}, match rate vs county median of {bedroom_col} = {match:.2%}')
    MED_BRMS_VERIFIED = match > 0.90
else:
    # Construct it
    if bedroom_col in master.columns:
        master['cty_med_bedrooms'] = master[COUNTY_COL].map(
            master.groupby(COUNTY_COL)[bedroom_col].median()
        )
        MED_BRMS_VERIFIED = True
        med_brms_col = 'cty_med_bedrooms'
        print('cty_med_bedrooms constructed from county median of', bedroom_col)
    else:
        MED_BRMS_VERIFIED = False
        print('Bedroom column not found; med_brms skipped.')

In [ ]:
# ── 5.3  Verification verdicts & decisions ────────────────────────────────
UNVERIFIED_EXCLUDE = []

# Drop unverified columns that would pollute the analytical frame
for c in UNVERIFIED_EXCLUDE:
    if c in master.columns:
        master = master.drop(columns=[c])
        print(f'Dropped (unverified provenance): {c}')

print('\nProvenance verdict summary:')
print(f'  prop_util      : {"VERIFIED" if PROP_UTIL_VERIFIED else "EXCLUDED"}')
print(f'  med_brms       : {"VERIFIED" if MED_BRMS_VERIFIED else "EXCLUDED"}')
print(f'  Remaining shape: {master.shape}')

---
## 6. General Imputation — Low & Moderate Tiers

Strategy: **stratified group-median** (by `urban_rural`) for numeric variables; **group-mode** for categorical/ordinal. Moderate-tier columns get an `_was_imputed` flag.

In [ ]:
# ── 6.1  Experiment: imputation strategy on c01_4 (water dist) ────────────
exp_col = 'c01_4' if 'c01_4' in master.columns else ('water_dist_mins' if 'water_dist_mins' in master.columns else None)
if exp_col:
    s = master[exp_col]
    orig_skew = skew(s.dropna())
    gm = s.fillna(master.groupby(URBAN_COL)[exp_col].transform('median'))
    print(f'{exp_col}: n_missing={s.isnull().sum()}, skew_orig={orig_skew:.2f}, skew_group_median={skew(gm):.2f}')
    print('Group-median preserves skew better than global median → confirmed strategy.')

In [ ]:
# ── 6.2  Apply stratified imputation ─────────────────────────────────────
# Exclude structurally-handled columns
ALREADY_HANDLED = (
    set(lp_cols) |
    {RENT_COL, 'rent_actual_kes', 'k05'} |
    set(mort_fin_cols) |
    set(UNVERIFIED_EXCLUDE)
)

low_mod_cols = tier_df[tier_df['tier'].isin(['Low', 'Moderate'])].index.tolist()
to_impute = [c for c in low_mod_cols
             if c in master.columns and c not in ALREADY_HANDLED]

imputation_log = []
for c in to_impute:
    pct = master[c].isnull().mean()
    if pct == 0:
        continue
    is_mod = pct > 0.20
    if is_mod:
        master[f'{c}_was_imputed'] = master[c].isnull().astype(int)

    if pd.api.types.is_numeric_dtype(master[c]):
        fill = master.groupby(URBAN_COL)[c].transform('median')
        master[c] = master[c].fillna(fill).fillna(master[c].median())
        method = 'group median (urban_rural)'
    else:
        fill = master.groupby(URBAN_COL)[c].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
        master[c] = master[c].fillna(fill)
        method = 'group mode (urban_rural)'

    imputation_log.append({
        'variable': c, 'tier': 'Moderate' if is_mod else 'Low',
        'pct_missing_before': round(pct * 100, 2), 'method': method,
    })

imp_log_df = pd.DataFrame(imputation_log)
print(f'Imputed {len(imp_log_df)} columns ({imp_log_df.tier.value_counts().to_dict()})')
print(imp_log_df.head(20))

---
## 7. Outlier Treatment — Winsorisation

Monetary survey variables have genuine right tails. We **winsorise at [1st, 99th pct] by urban/rural stratum** — no rows dropped, extreme values capped to preserve distributional shape while preventing a handful of outliers from dominating Shapiro-Wilk or correlation results.

In [ ]:
# ── 7.1  Compare outlier methods on a representative variable ─────────────
from scipy.stats import zscore

rep_col = EXP_COLS[0] if EXP_COLS else None
if rep_col:
    s = master[rep_col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    n_iqr  = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
    n_z3   = (np.abs(zscore(s)) > 3).sum()
    n_p99  = (s > s.quantile(0.99)).sum()
    print(f'{rep_col}: IQR-flags={n_iqr} ({n_iqr/len(s):.1%}),  Z>3={n_z3} ({n_z3/len(s):.1%}),  P99-above={n_p99}')
    print('IQR over-flags skewed expenditure → winsorisation at P99 is correct.')

In [ ]:
# ── 7.2  Apply: stratified winsorisation at [1st, 99th] pct ──────────────
MONETARY_COLS = [c for c in
    EXP_COLS + [RENT_COL, 'l13', 'mortgage_repayment_kes', 'l14',
                'dwelling_value_kes', 'l15', 'imputed_rent_kes',
                'c14_1', 'spend_water_kes', 'c14_2', 'spend_electricity_kes',
                'c14_3', 'spend_energy_other_kes', 'k10', 'rent_deposit_kes']
    if c in master.columns
]
MONETARY_COLS = list(set(MONETARY_COLS))

win_log = []
for c in MONETARY_COLS:
    lo = master.groupby(URBAN_COL)[c].transform(lambda x: x.quantile(0.01))
    hi = master.groupby(URBAN_COL)[c].transform(lambda x: x.quantile(0.99))
    n_lo = (master[c] < lo).sum()
    n_hi = (master[c] > hi).sum()
    master[c] = master[c].clip(lower=lo, upper=hi)
    win_log.append({'variable': c, 'n_lo': n_lo, 'n_hi': n_hi,
                    'pct_capped': round((n_lo+n_hi)/len(master)*100, 2)})

win_df = pd.DataFrame(win_log)
print(f'Winsorised {len(MONETARY_COLS)} monetary columns at [1st, 99th] pct by {URBAN_COL}.')
print(win_df)

In [ ]:
# ── 7.3  Count-variable sanity check ─────────────────────────────────────
count_cols = [c for c in ['hh_size', 'dw_rooms', 'd11', 'dw_bedrooms', 'd11_2', 'd12']
              if c in master.columns]
for c in count_cols:
    print(f'{c:<15}: min={master[c].min()}, max={master[c].max()}, p99={master[c].quantile(0.99):.0f}')

---
## 8. Consistency & Cross-Variable Logic Checks

In [ ]:
# ── 8.1  Cross-variable consistency checks ────────────────────────────────
consistency = []

# (a) Dwelling year built in the future
for c in ['l07', 'dwelling_yr_built']:
    if c in master.columns:
        n = (master[c] > 2026).sum()
        consistency.append((f'{c} > 2026', n))

# (b) Household size ≤ 0
size_col = 'hh_size' if 'hh_size' in master.columns else 'hhsize'
if size_col in master.columns:
    n = (master[size_col] <= 0).sum()
    consistency.append((f'{size_col} ≤ 0', n))

# (c) Any expenditure < 0 after winsorisation
n_neg = (master[EXP_COLS] < 0).any(axis=1).sum() if EXP_COLS else 0
consistency.append(('Any expenditure < 0 after winsorisation', n_neg))

# (d) Owner-occupier also paying rent
if OWNER_COL in master.columns and RENT_COL in master.columns:
    n = ((master[OWNER_COL] == 1) & (master[RENT_COL].fillna(0) > 0)).sum()
    consistency.append(('Owner-occupier (j04_1==1) with rent > 0', n))
    if n:
        master.loc[(master[OWNER_COL] == 1), RENT_COL] = np.nan
        CORRECTIONS_LOG.append((RENT_COL, 'owner-occupier rent → NaN', n))

# (e) Non-landowner with active land-parcel indicators
if LAND_COL in master.columns and 'lp_has_title' in master.columns:
    bad = (master[LAND_COL] == 0) & (~master['lp_has_title'].isin([0, 'No land']))
    consistency.append(('Non-landowner with lp_has_title active', bad.sum()))

consist_df = pd.DataFrame(consistency, columns=['check', 'n_flagged'])
print('Consistency check results:')
print(consist_df)

---
## 9. Feature Engineering

This section builds **all derived features** needed for both:
1. **HFVS pillar construction** (the five sub-scores, computed in the modeling notebook)
2. **Proxy feature matrix** (the 24 leakage-free features used by LightGBM)

Every variable engineered here is retained in `master_frame_clean.parquet`.

In [ ]:
# ── 9.1  P1 Financial Stress — total expenditure & rent burden ────────────
master['total_exp'] = master[EXP_COLS].sum(axis=1, skipna=True)
master['total_exp'] = master['total_exp'].replace(0, np.nan)

# Non-housing expenditure (denominator for rent burden)
non_housing = master['total_exp'] - master.get('g01h', pd.Series(0, index=master.index))
non_housing = non_housing.where(non_housing > 0, np.nan)

# Rent burden: only for renters
if RENT_COL in master.columns:
    master['rent_burden'] = np.where(
        master['is_renter'] == 1,
        (master[RENT_COL] / non_housing).clip(upper=5.0),  # cap at 500%
        np.nan
    )
    master['rent_burden'] = master['rent_burden'].clip(upper=1.0)
    master['rent_burdened'] = (
        master['rent_burden']
        .apply(lambda x: 1 if x > 0.30 else (0 if pd.notna(x) else pd.NA))
        .astype('Int64')
    )
    print(f'rent_burden: {master["rent_burden"].notna().sum():,} renters, '
          f'mean={master["rent_burden"].mean():.3f}')
    print(f'rent_burdened (>30%): {master["rent_burdened"].sum():,} households')

# Utility expenditure ratio
if util_cols:
    master['utility_exp'] = master[util_cols].sum(axis=1, skipna=True)
    master['utility_income_ratio'] = master['utility_exp'] / master['total_exp'].replace(0, np.nan)
print('P1 features built: total_exp, rent_burden, rent_burdened, utility_exp, utility_income_ratio')

In [ ]:
# ── 9.2  P2 Physical Quality — material scores & crowding ─────────────────
# Material coding: lower code = better material in the KHS codebook
# Normalise to [0,1] where 1 = worst (highest vulnerability)

def worst_material_flag(col_name, worst_codes):
    """Binary flag: 1 if code is in worst_codes (most vulnerable), 0 otherwise."""
    if col_name not in master.columns:
        return pd.Series(0.0, index=master.index)
    return master[col_name].isin(worst_codes).astype(float)

# Based on KHS codebook:
# Wall: 4=mud/earth, 5=wood/wattle, 6=bamboo → worst
# Roof: 3=grass/thatch, 4=makuti, 5=mud/dung → worst
# Floor: 3=earth/mud → worst
wall_poor  = worst_material_flag(WALL_COL,  [4, 5, 6])
roof_poor  = worst_material_flag(ROOF_COL,  [3, 4, 5])
floor_poor = worst_material_flag(FLOOR_COL, [3])

# Note: c04/toilet_type and c01_1/water_src_main also feed P2 but NOT re-used in P5
toilet_poor = worst_material_flag(TOILET_COL, [7, 8])   # pit no slab, open
water_poor  = worst_material_flag(WATER_COL,  [4, 9, 10, 11, 12])  # surface/unprotected

master['p2_wall_poor']   = wall_poor
master['p2_roof_poor']   = roof_poor
master['p2_floor_poor']  = floor_poor
master['p2_toilet_poor'] = toilet_poor
master['p2_water_poor']  = water_poor

# Composite physical quality score (higher = more vulnerable)
master['physical_quality_score'] = (
    0.25 * wall_poor + 0.25 * roof_poor + 0.25 * floor_poor +
    0.125 * toilet_poor + 0.125 * water_poor
)

# Crowding: persons per room
room_col = 'dw_rooms' if 'dw_rooms' in master.columns else ('d11' if 'd11' in master.columns else None)
size_col = 'hh_size'  if 'hh_size'  in master.columns else ('hhsize' if 'hhsize' in master.columns else None)
if room_col and size_col:
    master['persons_per_room'] = master[size_col] / master[room_col].replace(0, np.nan)
    master['persons_per_room'] = master['persons_per_room'].clip(upper=master['persons_per_room'].quantile(0.99))

# Subjective quality scores (h01-h11: 1=Good, 3=Poor → invert to vulnerability scale)
h_cols = [c for c in ['h01','h02','h03','h04','h05','h06','h07','h08','h09','h10','h11',
                        'perc_structure','perc_roof','perc_walls','perc_floor',
                        'perc_ventilation','perc_lighting','perc_water',
                        'perc_sanitation','perc_waste','perc_security','perc_overall']
          if c in master.columns]
if h_cols:
    # Invert: 1 (Good)→0, 2 (Fair)→0.5, 3 (Poor)→1
    subj = master[h_cols].apply(lambda s: (s - 1) / 2)
    master['subjective_quality_score'] = subj.mean(axis=1)
    if 'physical_quality_score' in master.columns:
        master['quality_gap'] = master['physical_quality_score'] - master['subjective_quality_score']

print(f'P2 features built: physical_quality_score, persons_per_room, subjective_quality_score, quality_gap')
print(master[['physical_quality_score','persons_per_room']].describe())

In [ ]:
# ── 9.3  P3 Tenure Security — composite score (higher = more insecure) ────
# no_title: 1 if no formal document
# eviction: 1 if at eviction risk
# no_land:  1 if does not own land
# is_renter: 1 if renting (included as mild insecurity signal)

no_title  = (1 - master[TITLE_COL].fillna(0)) if TITLE_COL in master.columns else pd.Series(0.5, index=master.index)
eviction  = master[EVICT_COL].fillna(0)        if EVICT_COL in master.columns else pd.Series(0, index=master.index)
no_land   = (1 - master[LAND_COL].fillna(0))  if LAND_COL  in master.columns else pd.Series(0.5, index=master.index)
is_renter_f = master['is_renter'].fillna(0)

master['tenure_security_score'] = (
    0.35 * no_title + 0.35 * eviction + 0.20 * no_land + 0.10 * is_renter_f
)
print(f'P3 tenure_security_score: min={master["tenure_security_score"].min():.3f}, '
      f'mean={master["tenure_security_score"].mean():.3f}, max={master["tenure_security_score"].max():.3f}')

In [ ]:
# ── 9.4  P4 Hazard Exposure — triple-exposure flag ────────────────────────
flood  = master[FLOOD_COL].fillna(0).clip(0, 2) / 2.0 if FLOOD_COL in master.columns else pd.Series(0, index=master.index)
slide  = master[ERODE_COL].fillna(0).clip(0, 2) / 2.0 if ERODE_COL in master.columns else pd.Series(0, index=master.index)
haz_z  = master[HAZ_COL].fillna(0)                     if HAZ_COL   in master.columns else pd.Series(0, index=master.index)

master['hazard_score'] = 0.40 * flood + 0.40 * slide + 0.20 * haz_z

# Triple-exposed: flood + informal tenure + no formal title
flood_flag    = master[FLOOD_COL].isin([1, 2]) if FLOOD_COL in master.columns else pd.Series(False, index=master.index)
informal_flag = master['is_renter'].eq(1)
no_title_flag = (master[TITLE_COL] == 0)       if TITLE_COL in master.columns else pd.Series(False, index=master.index)
master['triple_exposed'] = (flood_flag & informal_flag & no_title_flag).astype(int)

print(f'P4 hazard_score: mean={master["hazard_score"].mean():.3f}')
print(f'triple_exposed : {master["triple_exposed"].sum():,} households ({master["triple_exposed"].mean():.2%})')

In [ ]:
# ── 9.5  P5 Utility Deprivation — count of deprivation flags ─────────────
dep = pd.DataFrame(index=master.index)

# Water: surface / unprotected source
dep['water_deprived']    = worst_material_flag(WATER_COL,  [4, 9, 10, 11, 12])
# Sanitation: pit no slab / open
dep['sanit_deprived']    = worst_material_flag(TOILET_COL, [7, 8])
# Cooking: firewood / charcoal
dep['cooking_deprived']  = worst_material_flag(COOK_COL,   [7, 9])
# Lighting: kerosene / none
dep['lighting_deprived'] = worst_material_flag(LIGHT_COL,  [5, 6])
# Handwashing: no facility
if WASH_COL in master.columns:
    dep['wash_deprived'] = (master[WASH_COL] == 0).astype(float)
else:
    dep['wash_deprived'] = 0.0

master['utility_deprivation_score'] = dep.sum(axis=1) / 5.0  # normalise 0–1
for c in dep.columns:
    master[c] = dep[c]  # keep individual flags for EDA

print('P5 utility_deprivation_score distribution:')
print(master['utility_deprivation_score'].value_counts().sort_index())

In [ ]:
# ── 9.6  Demographic composites ───────────────────────────────────────────
# Dependency ratio — from individual aggregates
if {'n_children', 'n_elderly', 'n_working_age'}.issubset(master.columns):
    master['dependency_ratio'] = (
        (master['n_children'] + master['n_elderly'])
        / master['n_working_age'].replace(0, np.nan)
    )
    # Cap at 99th percentile to handle all-dependent households
    cap = master['dependency_ratio'].quantile(0.99)
    master['dependency_ratio'] = master['dependency_ratio'].clip(upper=cap).fillna(cap)
elif 'dependency_ratio' not in master.columns:
    # Approximate: children/elderly split not available — set to NaN
    master['dependency_ratio'] = np.nan

print(f'dependency_ratio: {master["dependency_ratio"].notna().sum():,} non-null')
if master['dependency_ratio'].notna().sum() > 0:
    print(master['dependency_ratio'].describe())

In [ ]:
# ── 9.7  Interaction terms for the proxy matrix ───────────────────────────
# urban_x_tenure: urban/rural × owner-occupier — key interaction per SHAP analysis
if URBAN_COL in master.columns and OWNER_COL in master.columns:
    master['urban_x_tenure'] = (
        master[URBAN_COL].fillna(0).astype(float) *
        master[OWNER_COL].fillna(0).astype(float)
    )

# size_x_utility: household size × utility deprivation
if size_col and 'utility_deprivation_score' in master.columns:
    master['size_x_utility'] = (
        master[size_col].fillna(master[size_col].median()) *
        master['utility_deprivation_score']
    )

print('Interaction terms built: urban_x_tenure, size_x_utility')

In [ ]:
# ── 9.8  Engineering summary ──────────────────────────────────────────────
ENGINEERED = [
    'total_exp', 'rent_burden', 'rent_burdened', 'utility_exp', 'utility_income_ratio',
    'p2_wall_poor', 'p2_roof_poor', 'p2_floor_poor', 'p2_toilet_poor', 'p2_water_poor',
    'physical_quality_score', 'persons_per_room',
    'subjective_quality_score', 'quality_gap',
    'tenure_security_score', 'hazard_score', 'triple_exposed',
    'utility_deprivation_score',
    'water_deprived', 'sanit_deprived', 'cooking_deprived', 'lighting_deprived', 'wash_deprived',
    'dependency_ratio', 'is_renter',
    'urban_x_tenure', 'size_x_utility',
]
ENGINEERED = [c for c in ENGINEERED if c in master.columns]

eng_summ = master[ENGINEERED].describe().T
eng_summ['pct_missing'] = (master[ENGINEERED].isnull().mean() * 100).round(1)
print(f'Engineered features: {len(ENGINEERED)}')
print(eng_summ[['count','mean','std','min','max','pct_missing']])

---
## 10. Column Inventory & Selection — Building the 25–35 Variable Frame

This is the critical design step. Every column in the final working subset is justified against one of four criteria:
- **Pillar ingredient**: directly used in HFVS sub-score formula
- **Pillar score**: engineered composite from pillar ingredients
- **Proxy feature**: leakage-free predictor in the LightGBM model
- **Administrative**: identifier, weight, stratum (not a feature)

In [ ]:
# ── 10.1  Define the final working subset ─────────────────────────────────
#
# DESIGN RATIONALE:
# The working subset must contain:
#   (A) All five pillar scores (engineered)
#   (B) All 24 proxy features (as defined in modeling notebook, Section 4.2)
#   (C) Key administrative columns (ID, weight, county, stratum)
#   (D) A small number of contextual / diagnostic columns
#
# FORMULA ANCESTORS (banned from proxy matrix — would be leakage if used as features):
#   P1: g01a–g01k, k05/rent_actual_kes, j09/housing_cost_burden, j10/missed_payment
#   P2: dw_wall_mat/d08, dw_roof_mat/d09, dw_floor_mat/d10, c04/toilet_type, c01_1/water_src_main
#   P3: j05/has_title_doc, j11/eviction_risk, i00/owns_land, j04_1/is_owner_occupier
#   P4: e06/flood_exposure, e07/erosion_exposure, d07/dw_in_hazard_zone
#   P5: c11/cooking_fuel, c10/lighting_src, c05/has_handwash_facility
#   Composites: physical_quality_score, tenure_security_score, etc.

# ── SECTION A: Administrative & stratum ──────────────────────────────────
id_col_name    = 'hh_id' if 'hh_id' in master.columns else 'interview__key'
ADMIN_COLS = [c for c in [id_col_name, COUNTY_COL, URBAN_COL, WEIGHT_COL] if c in master.columns]

# ── SECTION B: Five pillar scores (engineered) ───────────────────────────
PILLAR_SCORES = [c for c in [
    'total_exp',                    # P1 raw ingredient — kept for report EDA
    'rent_burden',                  # P1 core measure
    'rent_burdened',                # P1 binary threshold indicator
    'utility_income_ratio',         # P1 supplementary
    'physical_quality_score',       # P2 composite
    'persons_per_room',             # P2 crowding
    'subjective_quality_score',     # P2 subjective
    'quality_gap',                  # P2 objective-subjective gap
    'tenure_security_score',        # P3 composite
    'hazard_score',                 # P4 composite
    'triple_exposed',               # P4 extreme vulnerability flag
    'utility_deprivation_score',    # P5 composite
] if c in master.columns]

# ── SECTION C: Proxy feature matrix (24 variables, leakage-free) ─────────
# Matching exactly the PROXY_FEATURES list in the modeling notebook
PROXY_FEATURES = [c for c in [
    # Demographic
    'hh_size'     if 'hh_size'  in master.columns else 'hhsize',  # crowding proxy
    'max_edu_isced' if 'max_edu_isced' in master.columns else 'ken_edu_isced11',  # education
    'hh_head_sex'   if 'hh_head_sex'   in master.columns else 'b04',              # sex of head
    'mean_age'      if 'mean_age'       in master.columns else 'b05_years',        # age
    'dependency_ratio',                                                             # financial stress proxy
    'has_disability' if 'has_disability' in master.columns else 'any_disability',  # vulnerability amplifier

    # Geographic context
    URBAN_COL,     # already in ADMIN but also a feature
    COUNTY_COL,    # 47 counties — spatial context

    # Assets & financial inclusion
    'owns_mobile'  if 'owns_mobile'  in master.columns else 'c13__2',
    'owns_tv'      if 'owns_tv'      in master.columns else 'c13__3',
    'has_internet' if 'has_internet' in master.columns else 'internet',
    'owns_other_property' if 'owns_other_property' in master.columns else 'g04',

    # Housing characteristics (NOT formula ingredients)
    'dw_rooms'    if 'dw_rooms'    in master.columns else 'd11',    # room count (not quality)
    'dw_bedrooms' if 'dw_bedrooms' in master.columns else 'd11_2',  # bedrooms
    'dw_type'     if 'dw_type'     in master.columns else 'd03',    # dwelling type
    'hh_tenure_type' if 'hh_tenure_type' in master.columns else 'd01',  # tenure type flag

    # Stated housing problems (subjective, NOT measured)
    'prob_overcrowding' if 'prob_overcrowding' in master.columns else 'g05__1',
    'prob_poor_water'   if 'prob_poor_water'   in master.columns else 'g05__2',
    'prob_high_cost'    if 'prob_high_cost'    in master.columns else 'g05__7',

    # Stability & aspiration
    'yrs_in_dwelling'  if 'yrs_in_dwelling'  in master.columns else 'j12_1',
    'satisfied_tenure' if 'satisfied_tenure' in master.columns else 'j13',

    # Slum/informal settlement
    SLUM_COL,

    # County contextual (from county/mortgage aggregates)
    'mort_rate'    if 'mort_rate'    in master.columns else 'mort_interest_rate',
    'persons_per_room',  # crowding using room count (not quality-scored)
] if c and c in master.columns]

# Remove duplicates while preserving order
PROXY_FEATURES = list(dict.fromkeys(PROXY_FEATURES))

# Verify no leakage (proxy features must not be formula ancestors)
FORMULA_ANCESTORS = set(
    EXP_COLS +
    [RENT_COL, BURDEN_COL, MISSED_COL,
     WALL_COL, ROOF_COL, FLOOR_COL, TOILET_COL, WATER_COL,
     TITLE_COL, EVICT_COL, LAND_COL, OWNER_COL,
     FLOOD_COL, ERODE_COL, HAZ_COL,
     COOK_COL, LIGHT_COL, WASH_COL,
     'physical_quality_score', 'tenure_security_score',
     'hazard_score', 'utility_deprivation_score',
     'rent_burden', 'rent_burdened', 'total_exp']
)

LEAKED = set(PROXY_FEATURES) & FORMULA_ANCESTORS
if LEAKED:
    print(f'⚠  LEAKAGE DETECTED — removing from proxy matrix: {LEAKED}')
    PROXY_FEATURES = [f for f in PROXY_FEATURES if f not in LEAKED]

print(f'\nProxy feature matrix: {len(PROXY_FEATURES)} features (leakage check passed ✓)')
for f in PROXY_FEATURES:
    print(f'  {f}')

In [ ]:
# ── 10.2  Assemble the working subset & count ──────────────────────────────
WORKING_COLS = list(dict.fromkeys(
    ADMIN_COLS + PILLAR_SCORES + PROXY_FEATURES
))
WORKING_COLS = [c for c in WORKING_COLS if c in master.columns]

print(f'\n{'='*60}')
print(f'WORKING SUBSET: {len(WORKING_COLS)} columns × {master.shape[0]:,} rows')
print(f'{'='*60}')
print(f'  Administrative : {len(ADMIN_COLS)} columns')
print(f'  Pillar scores  : {len(PILLAR_SCORES)} columns')
print(f'  Proxy features : {len(PROXY_FEATURES)} columns')
print(f'  Total unique   : {len(WORKING_COLS)} columns')

if len(WORKING_COLS) < 25:
    print(f'\n⚠  WARNING: {len(WORKING_COLS)} < 25 columns. Checking for missing proxy features...')
    missing_proxy = [f for f in PROXY_FEATURES if f not in master.columns]
    print(f'   Proxy features not found in master: {missing_proxy}')
elif len(WORKING_COLS) > 35:
    print(f'\n⚠  WARNING: {len(WORKING_COLS)} > 35 columns. Consider trimming duplicate proxies.')
else:
    print(f'\n✓  Column count {len(WORKING_COLS)} is within the target range [25, 35].')

In [ ]:
# ── 10.3  Justification table — all working subset columns ────────────────
COL_ROLES = {}
for c in ADMIN_COLS:       COL_ROLES[c] = 'Administrative'
for c in PILLAR_SCORES:    COL_ROLES[c] = 'Pillar Score'
for c in PROXY_FEATURES:   COL_ROLES[c] = 'Proxy Feature'

COL_NOTES = {
    id_col_name: 'Unique household identifier',
    COUNTY_COL:  'County code (1–47); geographic unit for Spearman validation',
    URBAN_COL:   'Urban/Rural stratum; stratification variable throughout',
    WEIGHT_COL:  'Survey weight; used in weighted aggregations only (not a feature)',
    'total_exp': 'Sum of g01a–g01k; P1 denominator; included for report EDA',
    'rent_burden': 'k05 / non-housing expenditure; core P1 metric',
    'rent_burdened': 'Binary: rent_burden > 30%; P1 policy threshold indicator',
    'utility_income_ratio': 'Utility expenditure / total expenditure',
    'physical_quality_score': 'Weighted composite of wall/roof/floor/toilet/water quality',
    'persons_per_room': 'Household size / room count; crowding index',
    'subjective_quality_score': 'Mean of h01-h11 inverted to vulnerability scale',
    'quality_gap': 'Objective minus subjective quality; reveals hidden vulnerability',
    'tenure_security_score': 'Weighted composite: no title + eviction risk + no land + renter',
    'hazard_score': 'Flood + erosion + hazard zone exposure composite',
    'triple_exposed': 'Binary: flood-exposed AND renting AND no formal title',
    'utility_deprivation_score': 'Count of deprivation flags (water/sanit/cooking/lighting/wash) / 5',
    'dependency_ratio': 'Dependents / working-age; financial stress proxy (no expenditure used)',
    'is_renter': 'Derived from pays_rent; structural missingness flag for rent columns',
    SLUM_COL: 'KNBS slum/informal settlement classification',
    'urban_x_tenure': 'Interaction: urban_rural × owner-occupier',
    'size_x_utility': 'Interaction: household size × utility deprivation score',
}

roles_df = pd.DataFrame([
    {
        'column': c,
        'role': COL_ROLES.get(c, 'Proxy Feature'),
        'dtype': str(master[c].dtype),
        'pct_missing': round(master[c].isnull().mean() * 100, 1),
        'n_unique': master[c].nunique(dropna=True),
        'note': COL_NOTES.get(c, '—'),
    }
    for c in WORKING_COLS
])

print(roles_df.groupby('role')['column'].count().rename('n_columns'))
print()
print(roles_df[['column','role','dtype','pct_missing']].to_string(index=False))

In [ ]:
# ── 10.4  Explicitly justify EXCLUDED columns ─────────────────────────────
all_master_cols = set(master.columns)
working_set     = set(WORKING_COLS)
dropped_cols    = all_master_cols - working_set

EXCLUSION_REASONS = {
    'Formula ancestor (leakage)': list(FORMULA_ANCESTORS & all_master_cols),
    'Redundant: subsumed by pillar score': [
        'water_deprived', 'sanit_deprived', 'cooking_deprived',
        'lighting_deprived', 'wash_deprived',
        'p2_wall_poor', 'p2_roof_poor', 'p2_floor_poor', 'p2_toilet_poor', 'p2_water_poor',
    ],
    'High missingness, no structural meaning': [
        c for c in high_cols if c not in WORKING_COLS and c not in FORMULA_ANCESTORS
    ],
    'County-level aggregate (not HH-level)': [
        c for c in master.columns if c.startswith(('mort_','fin_','loan_','wsvc_','nema_','cty_'))
        and not c.endswith('_market_absent') and c not in WORKING_COLS
    ],
    'Survey admin / ID columns not needed': [
        c for c in master.columns
        if c.startswith(('interview__', 'survey_', 'sample_', 'selected_', 'hh_uuid'))
        and c not in WORKING_COLS
    ],
    'Aspiration / detail columns (not analytically primary)': [
        c for c in master.columns
        if c.startswith(('aspire_', 'barrier_', 'loan_barrier_', 'move_reason_',
                         'j16__', 'j19__', 'j22__', 'k26__', 'k30', 'k31',
                         'l16__', 'l22_1__', 'renovation_'))
        and c not in WORKING_COLS
    ],
    'Detail columns for renters only (subsumed by rent_burden)': [
        c for c in master.columns
        if c.startswith(('k0', 'k1', 'k2', 'k3')) and c not in WORKING_COLS
        and c != RENT_COL
    ],
}

excl_rows = []
for reason, cols in EXCLUSION_REASONS.items():
    for c in cols:
        if c in dropped_cols:
            excl_rows.append({'column': c, 'reason_excluded': reason})

excl_df = pd.DataFrame(excl_rows).drop_duplicates('column').sort_values('reason_excluded')
print(f'Columns excluded from working subset: {len(dropped_cols)}')
print(f'Justified with explicit reason:       {len(excl_df)}')
print()
print(excl_df.groupby('reason_excluded')['column'].count().rename('n_cols').to_string())

---
## 11. Post-Cleaning EDA

With the cleaned and engineered frame ready, explore the final working variables visually.

In [ ]:
# ── 11.1  Missingness: before vs after ────────────────────────────────────
wk_cols_before = [c for c in WORKING_COLS if c in miss_before.index]
wk_cols_after  = [c for c in WORKING_COLS if c in master.columns]
miss_after = (master[wk_cols_after].isnull().mean() * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
miss_before.reindex(wk_cols_before).dropna().sort_values('pct_missing', ascending=False).head(20)['pct_missing'].plot(
    kind='barh', ax=axes[0], color=RED, alpha=0.8)
axes[0].set_title('Working Subset — Missingness BEFORE Cleaning')
axes[0].invert_yaxis(); axes[0].set_xlabel('% missing')

miss_after.sort_values(ascending=False).head(20).plot(
    kind='barh', ax=axes[1], color=TEAL, alpha=0.8)
axes[1].set_title('Working Subset — Missingness AFTER Cleaning')
axes[1].invert_yaxis(); axes[1].set_xlabel('% missing')

plt.tight_layout()
plt.savefig(FIGS / 'missingness_working_subset.png', dpi=150)
plt.show()
print('Figure saved.')

In [ ]:
# ── 11.2  Distributions of pillar scores ──────────────────────────────────
pillar_plot = [c for c in [
    'rent_burden', 'physical_quality_score', 'tenure_security_score',
    'hazard_score', 'utility_deprivation_score', 'persons_per_room'
] if c in master.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, c in zip(axes.flat, pillar_plot):
    sns.histplot(master[c].dropna(), kde=True, ax=ax, color=TEAL, bins=40)
    ax.set_title(f'{c}\n(skew={skew(master[c].dropna()):.2f})')
    ax.set_xlabel('')
plt.suptitle('Pillar Score Distributions (Post-Cleaning)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'pillar_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11.3  Urban vs Rural splits across pillar scores ─────────────────────
if URBAN_COL in master.columns:
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    pillar_5 = [c for c in ['rent_burden', 'physical_quality_score',
                              'tenure_security_score', 'hazard_score',
                              'utility_deprivation_score'] if c in master.columns]
    labels = {1: 'Urban', 2: 'Rural'}
    for ax, c in zip(axes, pillar_5):
        for val, lbl in labels.items():
            subset = master.loc[master[URBAN_COL] == val, c].dropna()
            subset.plot(kind='density', ax=ax, label=lbl)
        ax.set_title(c.replace('_', '\n'), fontsize=9)
        ax.legend(fontsize=7)
        ax.set_xlabel('')
    plt.suptitle('Pillar Scores: Urban vs Rural Distributions', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGS / 'pillar_urban_rural_density.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 11.4  Rent burden distribution (renters only) ─────────────────────────
if 'rent_burden' in master.columns:
    renters = master.loc[master['is_renter'] == 1, 'rent_burden'].dropna()
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(renters, kde=True, ax=ax, color=AMBER, bins=40)
    ax.axvline(0.30, color=RED, linestyle='--', linewidth=2,
               label=f'30% threshold (n burdened={master["rent_burdened"].sum():,})')
    ax.set_title(f'Rent Burden Distribution (Renters Only, n={len(renters):,})')
    ax.set_xlabel('Rent as share of non-housing expenditure')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGS / 'rent_burden_distribution.png', dpi=150)
    plt.show()

In [ ]:
# ── 11.5  Correlation heatmap — pillar scores & selected proxies ──────────
corr_cols = [c for c in [
    'rent_burden', 'physical_quality_score', 'subjective_quality_score',
    'tenure_security_score', 'hazard_score', 'utility_deprivation_score',
    'quality_gap', 'dependency_ratio', 'persons_per_room', 'total_exp'
] if c in master.columns]

corr = master[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Pillar Scores & Key Proxies\n(Post-Cleaning)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / 'correlation_heatmap_pillar_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11.6  County-level weighted summary ───────────────────────────────────
if WEIGHT_COL in master.columns and 'total_exp' in master.columns:
    def weighted_mean(df, col, w=WEIGHT_COL):
        return np.average(df[col].dropna(), weights=df[w].reindex(df[col].dropna().index))

    county_summ = (
        master.groupby(COUNTY_COL)
        .apply(lambda g: pd.Series({
            'n_households': len(g),
            'wt_mean_total_exp': weighted_mean(g, 'total_exp'),
            'wt_mean_rent_burden': weighted_mean(g, 'rent_burden') if 'rent_burden' in g.columns else np.nan,
            'pct_slum': g[SLUM_COL].eq(1).mean() * 100 if SLUM_COL in g.columns else np.nan,
        }))
        .sort_values('wt_mean_total_exp', ascending=False)
    )

    print('County summary (top 10 by weighted total expenditure):')
    print(county_summ.head(10).round(1))

    fig, ax = plt.subplots(figsize=(10, 5))
    county_summ['wt_mean_total_exp'].head(15).plot(kind='barh', ax=ax, color=GREEN, alpha=0.8)
    ax.set_title('Top 15 Counties — Survey-Weighted Mean Total Expenditure')
    ax.invert_yaxis()
    ax.set_xlabel('KES (monthly)')
    plt.tight_layout()
    plt.savefig(FIGS / 'county_weighted_expenditure.png', dpi=150)
    plt.show()

---
## 12. Save Cleaned Data

Two outputs:
1. **`master_frame_clean.parquet`** — full cleaned frame (~443+ columns including all engineered features)
2. **`dsa8301_working_subset.parquet`** — the 25–35 variable analytical frame

In [ ]:
# ── 12.0  Final dtype enforcement ─────────────────────────────────────────
# Binary columns → Int64 (nullable integer, preserves NaN)
BINARY_COLS = [
    'is_renter', 'rent_burdened', 'triple_exposed',
    'water_deprived', 'sanit_deprived', 'cooking_deprived',
    'lighting_deprived', 'wash_deprived',
    'p2_wall_poor', 'p2_roof_poor', 'p2_floor_poor', 'p2_toilet_poor', 'p2_water_poor',
]
for c in BINARY_COLS:
    if c in master.columns:
        master[c] = master[c].astype('Int64')

# Continuous composites → float64
FLOAT_COLS = [
    'total_exp', 'rent_burden', 'utility_income_ratio', 'utility_exp',
    'physical_quality_score', 'persons_per_room',
    'subjective_quality_score', 'quality_gap',
    'tenure_security_score', 'hazard_score',
    'utility_deprivation_score', 'dependency_ratio',
    'urban_x_tenure', 'size_x_utility',
]
for c in FLOAT_COLS:
    if c in master.columns:
        master[c] = pd.to_numeric(master[c], errors='coerce').astype(float)

print('Dtype enforcement complete.')
print(master[WORKING_COLS].dtypes.value_counts())

In [ ]:
# ── 12.1  Save full cleaned master frame ──────────────────────────────────
CLEAN_PATH = PQ / 'master_frame_clean.parquet'
master.to_parquet(CLEAN_PATH)
print(f'Saved full cleaned frame: {master.shape} → {CLEAN_PATH}')

In [ ]:
# ── 12.2  Save DSA8301 working subset ─────────────────────────────────────
dsa_df = master[WORKING_COLS].copy()
WORK_PATH = PQ / 'dsa8301_working_subset.parquet'
dsa_df.to_parquet(WORK_PATH)

print(f'Working subset: {dsa_df.shape} → {WORK_PATH}')
print()
print('Final column list:')
for i, c in enumerate(WORKING_COLS, 1):
    print(f'  {i:2}. {c:<40} ({COL_ROLES.get(c, "proxy")})')
print(f'\n  Total: {len(WORKING_COLS)} columns (target: 25-35)')
dsa_df.head()

---
## 13. Preprocessing Summary Table (for the Report)

Required by the DSA 8301 workflow guide: every cleaning decision documented in one table.

In [ ]:
# ── 13.1  Assemble preprocessing summary ──────────────────────────────────
summary_rows = []

# Sentinel & plausibility corrections (Section 2)
for var, action, n in CORRECTIONS_LOG:
    summary_rows.append({
        'Section': '2 – Sentinel/Plausibility',
        'Issue': f'Invalid/implausible value in {var}',
        'Treatment': action,
        'Rows/Values Affected': n,
    })

# Structural missingness (Section 4)
summary_rows.append({
    'Section': '4 – Structural Missingness',
    'Issue': 'lp_* columns null for non-landowners (i00==0)',
    'Treatment': 'Encoded as 0 / "No land" (structural, not imputed)',
    'Rows/Values Affected': int(master[LAND_COL].eq(0).sum()) if LAND_COL in master.columns else 'n/a',
})
summary_rows.append({
    'Section': '4 – Structural Missingness',
    'Issue': f'{RENT_COL} null for non-renters',
    'Treatment': 'Left as NaN (structural); is_renter flag created',
    'Rows/Values Affected': int((master['is_renter'] == 0).sum()),
})
summary_rows.append({
    'Section': '4 – Structural Missingness',
    'Issue': 'mort_/fin_/loan_ county aggregates null (60–81%)',
    'Treatment': '_market_absent flags created; values filled with 0',
    'Rows/Values Affected': f'{len(mort_fin_cols)} columns',
})

# Provenance (Section 5)
summary_rows.append({
    'Section': '5 – Provenance Verification',
    'Issue': 'prop_util / med_brms / min_rent / med_prop — unverified provenance',
    'Treatment': f'Excluded: {UNVERIFIED_EXCLUDE}' if UNVERIFIED_EXCLUDE else 'Retained after verification',
    'Rows/Values Affected': '4 pre-computed columns reviewed',
})

# Imputation (Section 6)
for _, row in imp_log_df.iterrows():
    summary_rows.append({
        'Section': '6 – Imputation',
        'Issue': f"{row['variable']} — {row['pct_missing_before']}% missing ({row['tier']} tier)",
        'Treatment': row['method'],
        'Rows/Values Affected': f"{row['pct_missing_before']:.1f}% of rows",
    })

# Outliers (Section 7)
for _, row in win_df.iterrows():
    if row['n_lo'] + row['n_hi'] > 0:
        summary_rows.append({
            'Section': '7 – Outlier Treatment',
            'Issue': f"{row['variable']} — extreme values",
            'Treatment': f'Winsorised at [1st, 99th] pct by {URBAN_COL}',
            'Rows/Values Affected': f"{row['pct_capped']:.2f}% capped ({row['n_lo']+row['n_hi']})",
        })

# Consistency (Section 8)
for check, n in consistency:
    if n > 0:
        summary_rows.append({
            'Section': '8 – Consistency',
            'Issue': check,
            'Treatment': 'Reviewed and corrected where possible (see Section 8)',
            'Rows/Values Affected': n,
        })

# Feature engineering (Section 9)
summary_rows.append({
    'Section': '9 – Feature Engineering',
    'Issue': 'Pillar sub-scores and proxy features needed for HFVS and LightGBM',
    'Treatment': f'{len(ENGINEERED)} engineered features created (pillar composites, crowding, burden ratios, interactions)',
    'Rows/Values Affected': f'{master.shape[0]:,} rows',
})

cleaning_summary = pd.DataFrame(summary_rows)
cleaning_summary.to_csv(TABS / 'preprocessing_summary_v2.csv', index=False)
print(f'Cleaning summary: {len(cleaning_summary)} rows → preprocessing_summary_v2.csv')
cleaning_summary

In [ ]:
# ── 13.2  Save column justification table ─────────────────────────────────
roles_df.to_csv(TABS / 'working_subset_column_justification.csv', index=False)
excl_df.to_csv(TABS / 'excluded_columns_justification.csv', index=False)

print('Saved:')
print(f'  {TABS}/working_subset_column_justification.csv  ({len(roles_df)} rows)')
print(f'  {TABS}/excluded_columns_justification.csv       ({len(excl_df)} rows)')
print(f'  {TABS}/preprocessing_summary_v2.csv            ({len(cleaning_summary)} rows)')

In [ ]:
# ── 13.3  Final readiness check ───────────────────────────────────────────
print('=' * 65)
print('CLEANING & PREPARATION NOTEBOOK — FINAL STATUS')
print('=' * 65)
print()
print(f'Master frame (clean)  : {master.shape[0]:,} rows × {master.shape[1]} columns')
print(f'Working subset        : {dsa_df.shape[0]:,} rows × {dsa_df.shape[1]} columns')
print()
print('Five-Pillar Engineering:')
for p, c in [
    ('P1 Financial Stress    ', 'rent_burden'),
    ('P2 Physical Quality    ', 'physical_quality_score'),
    ('P3 Tenure Security     ', 'tenure_security_score'),
    ('P4 Hazard Exposure     ', 'hazard_score'),
    ('P5 Utility Deprivation ', 'utility_deprivation_score'),
]:
    if c in master.columns:
        n_valid = master[c].notna().sum()
        print(f'  {p}: {n_valid:,} non-null ({n_valid/len(master):.1%})')
    else:
        print(f'  {p}: ✗ NOT FOUND')
print()
print(f'Proxy features (no leakage): {len(PROXY_FEATURES)}')
print(f'Leakage check passed       : ✓ ({len(set(PROXY_FEATURES) & FORMULA_ANCESTORS)} overlaps → 0)')
print()
print('Output files ready for DSA8301_KHS_modeling_evaluation.ipynb:')
print(f'  ✓ {PQ}/master_frame_clean.parquet')
print(f'  ✓ {PQ}/dsa8301_working_subset.parquet')
print()
print('Next step: load master_frame_clean.parquet in the modeling notebook')
print('  and proceed from Section 2 (HFVS target construction).')